# Replication of Buehler et al. (2019), *Deep Hedging* (Heston case)

Buehler, Gonon, Teichmann and Wood replace the analytic hedging rule with a
network trained to minimise a convex risk measure of the terminal P&L. The
committed numbers live in `submission/data/deep_hedging/results.json`, produced
by `scripts/deep_hedging_replication.py` (same entry point as `REPORT.md` §6).

Default below: **load artifacts** (fast). Set `RUN_TRAINING = True` only for a
small smoke re-run; full reproduction uses the script with `--epochs 100`.


In [ ]:
import json
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from hedging import bench as B

RUN_TRAINING = False
DATA = ROOT / "submission" / "data" / "deep_hedging" / "results.json"
FIG = ROOT / "submission" / "data" / "deep_hedging" / "sharpe_vs_cost.png"

BASE = B.BenchConfig()
print("frozen bench (for context):", BASE.model, "cost=", BASE.cost)
print("artifact:", DATA.relative_to(ROOT))

## 1. Frictionless delta check (committed run)

With zero costs the network should track Black-Scholes delta. `REPORT.md` §4
records correlation **0.9907** on the full script run; we display the committed
JSON rather than re-train here.


In [ ]:
payload = json.loads(DATA.read_text())
fr = payload["frictionless"]
pl.DataFrame([{
    "correlation": fr["correlation"],
    "slope": fr["slope"],
    "rmse": fr["rmse"],
}])

## 2. Heston with proportional costs (committed run)

Sharpe ratios vs cost for learned hedger, every-step BS delta (Buehler baseline),
and the frozen band (`load_classical_best()`). Numbers match `REPORT.md` §2 table.


In [ ]:
rows = []
for block in payload["costs"]:
    s = block["summary"]
    rows.append({
        "cost_bp": block["cost"] * 1e4,
        "learned_sharpe": s["deep_sharpe"]["mean"],
        "bs_delta_sharpe": s["bs_delta_sharpe"]["mean"],
        "band_sharpe": s["band_sharpe"]["mean"],
        "edge_vs_band": s["edge_vs_band"]["mean"],
    })
table = pl.DataFrame(rows)
table

In [ ]:
if FIG.is_file():
    img = plt.imread(FIG)
    plt.figure(figsize=(7, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Committed figure (deep_hedging_replication.py)")
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    for col, label in (
        ("learned_sharpe", "learned"),
        ("bs_delta_sharpe", "BS delta"),
        ("band_sharpe", "frozen band"),
    ):
        ax.plot(table["cost_bp"], table[col], marker="o", label=label)
    ax.set_xlabel("transaction cost (bp)")
    ax.set_ylabel("Sharpe ratio")
    ax.legend()
    plt.show()

### Optional smoke re-run (off by default)

Mirrors the frictionless block in `scripts/deep_hedging_replication.py` with a
tiny budget. Do not cite smoke numbers as report figures.


In [ ]:
if not RUN_TRAINING:
    print("Set RUN_TRAINING = True to execute the smoke training cell.")
else:
    import torch
    from dataclasses import replace

    from hedging import bs, deep
    from hedging.hyper import Hyper

    cfg0 = replace(BASE, cost=0.0, n_val_paths=2_000)
    hyper = Hyper(epochs=20, hidden=64, depth=3, alpha=0.05, torch_seeds=(0,))
    paths = cfg0.simulate(cfg0.model, 2001, 2_000)
    tensor = torch.as_tensor(paths, dtype=torch.float32)
    nets, _ = B.train_ensemble(cfg0, hyper)
    learned = deep.hedge_positions(tensor, nets[0], cfg0.K, cfg0.dt, cfg0.T)
    n_steps = paths.shape[1] - 1
    target = np.stack([
        bs.bs_delta(paths[:, j], cfg0.K, max(cfg0.T - j * cfg0.dt, 0.0), 0.0, cfg0.sigma_impl, "call")
        for j in range(n_steps)
    ], axis=1)
    x, y = learned.reshape(-1), target.reshape(-1)
    print("smoke correlation:", round(float(np.corrcoef(x, y)[0, 1]), 4))